In [1]:
import os
import numpy as np
import rasterio
from PIL import Image

def slice_geotiff(geotiff_path, output_dir, output_size, slice_flag=True, to_jpeg=True):
    """
    Slice a GeoTIFF color image into smaller square images or convert it to a single JPEG.

    Args:
        geotiff_path (str): Path to the input GeoTIFF file.
        output_dir (str): Directory to save the output slices or image.
        output_size (int): Size (in pixels) of each square output slice.
        slice_flag (bool): If True, slices the image into square tiles. If False, no slicing is performed.
        to_jpeg (bool): If True, outputs JPEG images; otherwise, retains TIFF format.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    with rasterio.open(geotiff_path) as src:
        img_array = src.read()  # Read all bands (multi-band support)
        height, width = img_array.shape[1], img_array.shape[2]
        
        # Determine the number of tiles in each dimension
        num_tiles_x = int(np.ceil(width / output_size))
        num_tiles_y = int(np.ceil(height / output_size))
        
        if not slice_flag:
            # Save the entire image as a single output
            img = np.moveaxis(img_array, 0, -1)  # Convert to HxWxBands
            img = Image.fromarray(img)
            if img.mode == 'RGBA' and to_jpeg:
                img = img.convert('RGB')  # Convert to RGB for JPEG format
            output_format = "JPEG" if to_jpeg else "TIFF"
            img.save(os.path.join(output_dir, f"output.{output_format.lower()}"), output_format)
            return
        
        # Slice into smaller square images
        for y in range(num_tiles_y):
            for x in range(num_tiles_x):
                # Calculate slice boundaries
                x_start = x * output_size
                y_start = y * output_size
                x_end = min(x_start + output_size, width)
                y_end = min(y_start + output_size, height)
                
                # Extract slice and pad with black pixels if necessary
                slice_img = img_array[:, y_start:y_end, x_start:x_end]
                pad_x = output_size - (x_end - x_start)
                pad_y = output_size - (y_end - y_start)
                
                if pad_x > 0 or pad_y > 0:
                    slice_img = np.pad(slice_img, ((0, 0), (0, pad_y), (0, pad_x)), mode='constant', constant_values=0)
                
                # Convert to image and save
                img = np.moveaxis(slice_img, 0, -1)
                img = Image.fromarray(img)
                if img.mode == 'RGBA' and to_jpeg:
                    img = img.convert('RGB')  # Convert to RGB for JPEG format
                filename = f"slice_{y}_{x}.jpg" if to_jpeg else f"slice_{y}_{x}.tif"
                img.save(os.path.join(output_dir, filename), "JPEG" if to_jpeg else "TIFF")


# def reverse_slicing(slice_dir, output_path, output_size):
#     """
#     Reassemble square image slices into a single image, discarding black pixels if needed.

#     Args:
#         slice_dir (str): Directory containing image slices.
#         output_path (str): Path to save the reassembled image.
#         output_size (int): Size (in pixels) of each square slice.
#     """
#     # Load all slice filenames and sort them to maintain order
#     slice_files = sorted([f for f in os.listdir(slice_dir) if f.endswith(('.jpg', '.tif'))])
    
#     # Determine the number of tiles in each dimension
#     num_tiles = int(np.sqrt(len(slice_files)))
#     assembled_img = None

#     # Get extension from the first slice to determine output format
#     if slice_files:
#         _, ext = os.path.splitext(slice_files[0])
#     else:
#         raise ValueError("No slice images found in the specified directory.")

#     # Determine the correct format for Pillow based on file extension
#     output_format = "JPEG" if ext.lower() == ".jpg" else "TIFF" if ext.lower() == ".tif" else ext[1:].upper()
#     output_path_f = f"{output_path}{ext}"

#     for i, file in enumerate(slice_files):
#         # Load each slice
#         img = Image.open(os.path.join(slice_dir, file))
#         img_array = np.array(img)
        
#         # Remove black padding
#         non_black_pixels = np.any(img_array != 0, axis=-1)
#         img_array = img_array[non_black_pixels.any(1)][:, non_black_pixels.any(0)]
        
#         # Append slice to final assembled image
#         if i % num_tiles == 0:
#             # Start a new row
#             row_img = img_array
#         else:
#             # Append to the current row
#             row_img = np.concatenate((row_img, img_array), axis=1)
        
#         if (i + 1) % num_tiles == 0:
#             # Complete row and add to assembled image
#             if assembled_img is None:
#                 assembled_img = row_img
#             else:
#                 assembled_img = np.concatenate((assembled_img, row_img), axis=0)
    
#     # Save the final assembled image
#     final_img = Image.fromarray(assembled_img)
#     if final_img.mode == 'RGBA' and output_format == "JPEG":
#         final_img = final_img.convert('RGB')  # Convert to RGB for JPEG
#     final_img.save(output_path_f, output_format)


import os
import numpy as np
from PIL import Image

def reverse_slicing(slice_dir, output_path, output_size):
    """
    Reassemble square image slices into a single image, discarding black pixels if needed.

    Args:
        slice_dir (str): Directory containing image slices.
        output_path (str): Path to save the reassembled image.
        output_size (int): Size (in pixels) of each square slice.
    """
    # Load all slice filenames and sort them to maintain order
    slice_files = sorted([f for f in os.listdir(slice_dir) if f.endswith(('.jpg', '.tif'))])
    
    # Determine the number of tiles in each dimension
    num_tiles = int(np.sqrt(len(slice_files)))
    
    # Open a sample slice to determine the tile size
    img_sample = Image.open(os.path.join(slice_dir, slice_files[0]))
    tile_height, tile_width = img_sample.size
    
    # Create an output file to write row by row
    _, ext = os.path.splitext(slice_files[0])
    output_format = "JPEG" if ext.lower() == ".jpg" else "TIFF" if ext.lower() == ".tif" else ext[1:].upper()
    output_path_f = f"{output_path}{ext}"
    
    # Create an empty file in the desired format
    mode = "RGB"  # Use RGB mode for both JPEG and TIFF
    
    # Create the initial blank output image as a file
    with Image.new(mode, (tile_width * num_tiles, tile_height * num_tiles)) as final_img:
        for row in range(num_tiles):
            row_img = None
            for col in range(num_tiles):
                # Load each slice
                slice_index = row * num_tiles + col
                img = Image.open(os.path.join(slice_dir, slice_files[slice_index]))
                img_array = np.array(img)
                
                # Remove black padding if present
                non_black_pixels = np.any(img_array != 0, axis=-1)
                img_array = img_array[non_black_pixels.any(1)][:, non_black_pixels.any(0)]
                
                # Concatenate slices horizontally to form the row
                if row_img is None:
                    row_img = img_array
                else:
                    row_img = np.concatenate((row_img, img_array), axis=1)
            
            # Convert the row to an image and paste it into the final image file
            row_img_pil = Image.fromarray(row_img)
            final_img.paste(row_img_pil, (0, row * tile_height))
        
        # Remove black pixels from the final image
        final_img_array = np.array(final_img)
        non_black_pixels = np.any(final_img_array != 0, axis=-1)
        final_img_array = final_img_array[non_black_pixels.any(1)][:, non_black_pixels.any(0)]
        
        # Save the completed image
        final_img = Image.fromarray(final_img_array)
        final_img.save(output_path_f, output_format, compression=None)  # Save without compression


In [4]:
import os

# Define paths and parameters
geotiff_path = "m_3008501_ne_16_030_20211103.tif"  # Path to the input GeoTIFF file
output_dir = "output_slices_geo"           # Directory to save sliced images
reassembled_output_path = "reassembled_output_tiff_3"  # Path for the reassembled image

output_size = 3000  # Size of each square slice in pixels
slice_flag = True  # Enable slicing

# Step 1: Slice the GeoTIFF image into tiles
print("Slicing the GeoTIFF image...")
slice_geotiff(
    geotiff_path=geotiff_path,
    output_dir=output_dir,
    output_size=output_size,
    slice_flag=slice_flag,  # Enable slicing
    to_jpeg=False            # Save slices as JPEG images
)

# print("Reassembling the image from slices...")
# reverse_slicing(
#     slice_dir=output_dir,
#     output_path=reassembled_output_path,
#     output_size=output_size  # The same size used for slicing
# )
# print(f"Reassembled image saved as {reassembled_output_path}")


Slicing the GeoTIFF image...
